In [1]:
# Define metadata for each table
metadata = [
    {
        "table_name": "dim_patient",
        "columns": [
            {"name": "patient_sk", "description": "Surrogate key"},
            {"name": "patient_id", "description": "From EHR"},
            {"name": "full_name", "description": "First + Last"},
            {"name": "gender", "description": "M/F/Other"},
            {"name": "dob", "description": "Date of birth"},
            {"name": "city", "description": "Location"},
            {"name": "state", "description": "Location"},
            {"name": "zip", "description": "Location"},
            {"name": "age_group", "description": "Derived (0–17, 18–40, etc.)"},
            {"name": "effective_from", "description": "SCD Type 2 tracking"},
            {"name": "effective_to", "description": "SCD Type 2 tracking"},
            {"name": "current_flag", "description": "SCD Type 2 tracking"}
        ]
    },
    {
        "table_name": "dim_provider",
        "columns": [
            {"name": "provider_sk", "description": "Surrogate key"},
            {"name": "provider_id", "description": "From EHR"},
            {"name": "provider_name", "description": "Full name"},
            {"name": "specialty", "description": "e.g., Cardiologist"},
            {"name": "npi_number", "description": "Unique ID"},
            {"name": "facility_name", "description": "Hospital/Clinic"},
            {"name": "effective_from", "description": "SCD Type 2 tracking"},
            {"name": "effective_to", "description": "SCD Type 2 tracking"},
            {"name": "current_flag", "description": "SCD Type 2 tracking"}
        ]
    },
    {
        "table_name": "dim_payer",
        "columns": [
            {"name": "payer_sk", "description": "Surrogate key"},
            {"name": "payer_id", "description": "From Insurance system"},
            {"name": "payer_name", "description": ""},
            {"name": "plan_type", "description": "Medicare, Medicaid, PPO, etc."},
            {"name": "payer_category", "description": "Government / Commercial"},
            {"name": "state", "description": "Operating region"},
            {"name": "effective_from", "description": "SCD Type 2 tracking"},
            {"name": "effective_to", "description": "SCD Type 2 tracking"},
            {"name": "current_flag", "description": "SCD Type 2 tracking"}
        ]
    },
    {
        "table_name": "dim_encounter",
        "columns": [
            {"name": "encounter_sk", "description": "Surrogate key"},
            {"name": "encounter_id", "description": "From EHR"},
            {"name": "encounter_type", "description": "Inpatient / Outpatient"},
            {"name": "visit_date_sk", "description": "FK → dim_date"},
            {"name": "provider_sk", "description": "FK → dim_provider"},
            {"name": "patient_sk", "description": "FK → dim_patient"}
        ]
    },
    {
        "table_name": "dim_date",
        "columns": [
            {"name": "date_sk", "description": "Surrogate key"},
            {"name": "date", "description": "Calendar date"},
            {"name": "day", "description": "Standard breakdown"},
            {"name": "month", "description": "Standard breakdown"},
            {"name": "quarter", "description": "Standard breakdown"},
            {"name": "year", "description": "Standard breakdown"},
            {"name": "day_of_week", "description": "Monday, Tuesday…"},
            {"name": "is_weekend", "description": "Y/N"},
            {"name": "is_holiday", "description": "Y/N"}
        ]
    }
]

In [ ]:
from langchain_openai import OpenAIEmbeddings
import  os
from dotenv import load_dotenv
load_dotenv()

import faiss
import numpy as np
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


# Flatten column metadata
column_metadata = []
for table in metadata:
    for col in table["columns"]:
        column_metadata.append({
            "table": table["table_name"],
            "name": col["name"],
            "description": col["description"]
        })

# Embed descriptions
embedder = OpenAIEmbeddings()
vectors = [embedder.embed_query(col["description"] or col["name"]) for col in column_metadata]

# Build FAISS index
dimension = len(vectors[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(vectors))

# Save metadata alongside index
faiss_db = {
    "index": index,
    "metadata": column_metadata
}

faiss.write_index(faiss_db["index"], "faiss_index.bin")

In [11]:
index = faiss.read_index("faiss_index.bin")

In [12]:
import pandas as pd

In [13]:
df = pd.read_csv(r"D:\Ascent_projects\Hack_etl\Data\source\clientA.csv")
df

,clm_no,submission_dt,total_amt,amt_paid,reject_reason,insurance,prv_cd,pt_ref,state
0,C1234,8/10/2024,450,400,NaN,Aetna,PR567,PT432,Processed
1,C1235,8/11/2024,700,700,NaN,UnitedHealth,PR568,PT433,Paid
2,C1236,8/12/2024,1200,0,Missing diagnosis code,BlueCross,PR569,PT434,Denied
3,C1237,8/12/2024,900,850,NaN,Cigna,PR570,PT435,Paid
4,C1238,8/13/2024,500,0,Duplicate submission,Humana,PR571,PT436,Denied
5,C1239,8/13/2024,350,350,NaN,Aetna,PR567,PT437,Paid
6,C1240,8/14/2024,780,760,NaN,UnitedHealth,PR568,PT438,Processed
7,C1241,8/14/2024,650,0,Service not covered,BlueCross,PR569,PT439,Denied
8,C1242,8/15/2024,300,300,NaN,Cigna,PR570,PT440,Paid
9,C1243,8/15/2024,1000,950,NaN,Humana,PR571,PT441,Processed


In [14]:
def match_column(col_name, faiss_db, threshold=0.85):
    vec = embedder.embed_query(col_name)
    D, I = faiss_db["index"].search(np.array([vec]), k=1)
    if D[0][0] >= threshold:
        return faiss_db["metadata"][I[0][0]]
    return None


In [15]:
for col in df.columns:
    match = match_column(col, faiss_db)
    if match:
        print(f"{col} → {match['name']} ({match['description']})")
    else:
        print(f"{col} → No match found")


clm_no → No match found
submission_dt → No match found
total_amt → No match found
amt_paid → No match found
reject_reason → No match found
insurance → No match found
prv_cd → No match found
pt_ref → No match found
state → No match found


In [16]:
import json

# Save metadata to disk
with open("column_metadata.json", "w") as f:
    json.dump(faiss_db["metadata"], f, indent=2)

In [18]:
from openai import OpenAI
import pandas as pd
import json

# Load CSV and metadata
with open("column_metadata.json") as f:
    metadata = json.load(f)

# Prepare prompt
csv_columns = df.columns.tolist()
metadata_text = "\n".join([f"{col['name']}: {col['description']}" for col in metadata])

prompt = f"""
You are a schema mapping assistant.

Here is the metadata schema:
{metadata_text}

Here are the columns from a new CSV file:
{csv_columns}

Please map each CSV column to the closest metadata column name, and explain your reasoning.
"""

# Call GPT-4
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

1. 'clm_no' to 'encounter_id': This mapping is inferred because a claim number in a health setting can be considered as a unique id for that specific encounter.

2. 'submission_dt' to 'date': This mapping is inferred because submission date can be considered as calendar date.

3. 'total_amt' (no direct correspondence): This column doesn't directly map to any existing schema column. This probably represents the total amount charged for a healthcare service but there is no similar field in the current schema.

4. 'amt_paid' (no direct correspondence): Similarly to 'total_amt', this column doesn't directly map to any existing schema. It probably stands for the amount that was paid for a healthcare service.

5. 'reject_reason' (no direct correspondence): This probably indicates the reason for a claim being rejected. However, the current schema doesn't track this information.

6. 'insurance' to 'payer_name': This mapping is inferred because insurance might be the name of the payer in terms 